# 01 - 环境搭建与模型准备

在DGX Spark上完成环境配置、依赖安装和基座模型下载。

**DGX Spark硬件**: GB10 Grace Blackwell | 128GB Unified Memory | 1 PFLOPS FP4

## 1.1 检查硬件环境

In [1]:
# 检查GPU和内存
import torch
import os

print("="*60)
print("DGX Spark 硬件检查")
print("="*60)
print(f"PyTorch版本: {torch.__version__}")
print(f"CUDA可用: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA版本: {torch.version.cuda}")
    print(f"GPU数量: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {props.name}")
        print(f"  总内存: {props.total_memory / 1024**3:.1f} GB")
    print(f"当前内存占用: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

# 检查CPU内存
import psutil
mem = psutil.virtual_memory()
print(f"CPU总内存: {mem.total / 1024**3:.1f} GB")
print(f"可用内存: {mem.available / 1024**3:.1f} GB")
print("="*60)

DGX Spark 硬件检查
PyTorch版本: 2.12.0+cu130
CUDA可用: True
CUDA版本: 13.0
GPU数量: 1
GPU 0: NVIDIA GB10
  总内存: 121.7 GB
当前内存占用: 0.00 GB
CPU总内存: 121.7 GB
可用内存: 118.9 GB


## 1.2 安装依赖包

In [ ]:
# 安装依赖 (仅需运行一次)
# 与项目 requirements.txt 对齐 — DGX Spark ARM64 实测版本见文件内注释
# (旧 pin: transformers==4.45.0/peft==0.12.0/bitsandbytes==0.43.3/trl==0.9.6 已废弃,
#  bitsandbytes 0.43.3 无 ARM64 wheel, 且 Qwen3.6 需要 transformers>=5.2.0)
!pip install -q -r ../requirements.txt

# 验证安装
import torch
import transformers
import peft
import trl
import datasets
print(f"torch: {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"peft: {peft.__version__}")
print(f"trl: {trl.__version__}")
print(f"datasets: {datasets.__version__}")
print("\n所有依赖安装完成!")

## 1.2b 预飞行检查

验证环境是否满足运行要求，包括：
- 必要 Python 包已安装且版本正确
- 必要目录已创建
- 配置文件可正常导入

In [3]:
import sys
from packaging import version

def preflight_check():
    """预飞行检查：验证环境 readiness"""
    errors = []
    warnings_list = []

    # 1. 检查必要包版本
    required_packages = {
        'transformers': '4.45.0',
        'peft': '0.12.0',
        'bitsandbytes': '0.43.3',
        'datasets': '2.21.0',
        'trl': '0.9.6',
        'accelerate': '0.33.0',
    }

    print("=" * 60)
    print("预飞行检查")
    print("=" * 60)

    print("\n[1/4] 检查必要包...")
    for pkg_name, min_ver in required_packages.items():
        try:
            mod = __import__(pkg_name)
            actual_ver = getattr(mod, "__version__", "unknown")
            if actual_ver != "unknown":
                if version.parse(actual_ver) < version.parse(min_ver):
                    errors.append(f"{pkg_name} 版本过低: {actual_ver} < {min_ver}")
                    print(f"  FAIL {pkg_name}: {actual_ver} < {min_ver}")
                else:
                    print(f"  PASS {pkg_name}: {actual_ver} (>= {min_ver})")
            else:
                warnings_list.append(f"{pkg_name}: 无法获取版本")
                print(f"  WARN {pkg_name}: 无法获取版本")
        except ImportError:
            errors.append(f"未安装包: {pkg_name}")
            print(f"  FAIL {pkg_name}: 未安装")

    # 2. 检查可选包（用于 02b）
    print("\n[2/4] 检查可选包（合成数据生成需要）...")
    optional_packages = {
        'openai': '1.35.0',
        'rouge_score': '0.1.2',
    }
    for pkg_name, min_ver in optional_packages.items():
        try:
            mod = __import__(pkg_name)
            actual_ver = getattr(mod, "__version__", "unknown")
            print(f"  PASS {pkg_name}: {actual_ver}")
        except ImportError:
            print(f"  SKIP {pkg_name}: 未安装（运行 02b 前需要安装）")

    # 3. 检查目录
    print("\n[3/4] 检查必要目录...")
    required_dirs = [
        '/home/meerkat/mongoose_ai/data',
        '/home/meerkat/mongoose_ai/models',
        '/home/meerkat/mongoose_ai/outputs',
        '/home/meerkat/mongoose_ai/checkpoints',
        '/home/meerkat/mongoose_ai/results',
    ]
    for d in required_dirs:
        import os
        if os.path.exists(d):
            print(f"  PASS {d}")
        else:
            os.makedirs(d, exist_ok=True)
            print(f"  CREATE {d} (已自动创建)")

    # 4. 检查配置文件
    print("\n[4/4] 检查配置文件...")
    sys.path.append('/home/meerkat/mongoose_ai')
    try:
        from config import (
            BASE_MODEL, MODELS_DIR, DATA_DIR, OUTPUTS_DIR,
            QLORA_CONFIG, DATA_CONFIG, BENCHMARK_CONFIG, HARDWARE_CONFIG
        )
        print(f"  PASS config.py: BASE_MODEL={BASE_MODEL}")
        # 验证关键配置值
        lora_dropout = QLORA_CONFIG['lora']['lora_dropout']
        if lora_dropout != 0.0:
            warnings_list.append(f"lora_dropout={lora_dropout} (建议 0.0 以兼容 MoE)")
            print(f"  WARN lora_dropout={lora_dropout} (建议 0.0)")
        else:
            print(f"  PASS lora_dropout=0.0")

        target_modules = QLORA_CONFIG['lora']['target_modules']
        if isinstance(target_modules, list) and len(target_modules) == 12:
            print(f"  PASS target_modules: {len(target_modules)} 个模块")
        else:
            warnings_list.append(f"target_modules 数量异常: {len(target_modules) if isinstance(target_modules, list) else "非列表"}")
            print(f"  WARN target_modules 数量异常")

    except Exception as e:
        errors.append(f"配置文件导入失败: {e}")
        print(f"  FAIL config.py: {e}")

    # 总结
    print("\n" + "=" * 60)
    if errors:
        print(f"预飞行检查失败: {len(errors)} 个错误")
        for e in errors:
            print(f"  ERROR: {e}")
        print("=" * 60)
        raise RuntimeError("预飞行检查失败，请修复上述错误后再继续")
    else:
        print("预飞行检查通过!")
        if warnings_list:
            print(f"警告: {len(warnings_list)} 个")
            for w in warnings_list:
                print(f"  WARN: {w}")
        print("=" * 60)

# 运行检查
preflight_check()

预飞行检查

[1/4] 检查必要包...
  PASS transformers: 5.10.1 (>= 4.45.0)
  PASS peft: 0.19.1 (>= 0.12.0)
  PASS bitsandbytes: 0.49.2 (>= 0.43.3)
  PASS datasets: 4.8.5 (>= 2.21.0)
  PASS trl: 1.5.1 (>= 0.9.6)
  PASS accelerate: 1.13.0 (>= 0.33.0)

[2/4] 检查可选包（合成数据生成需要）...
  PASS openai: 1.35.0
  PASS rouge_score: unknown

[3/4] 检查必要目录...
  PASS /home/meerkat/mongoose_ai/data
  PASS /home/meerkat/mongoose_ai/models
  PASS /home/meerkat/mongoose_ai/outputs
  PASS /home/meerkat/mongoose_ai/checkpoints
  PASS /home/meerkat/mongoose_ai/results

[4/4] 检查配置文件...
  PASS config.py: BASE_MODEL=Qwen/Qwen3.6-35B-A3B
  PASS lora_dropout=0.0
  PASS target_modules: 12 个模块

预飞行检查通过!


## 1.3 加载配置文件

In [4]:
import sys
sys.path.append('/home/meerkat/mongoose_ai')

from config import (
    BASE_MODEL, MODELS_DIR, DATA_DIR, OUTPUTS_DIR,
    QLORA_CONFIG, DATA_CONFIG, BENCHMARK_CONFIG, HARDWARE_CONFIG
)

print("="*60)
print("配置信息")
print("="*60)
print(f"基座模型: {BASE_MODEL}")
print(f"模型目录: {MODELS_DIR}")
print(f"数据目录: {DATA_DIR}")
print(f"输出目录: {OUTPUTS_DIR}")
print(f"\nQLoRA配置:")
print(f"  rank: {QLORA_CONFIG['lora']['r']}")
print(f"  alpha: {QLORA_CONFIG['lora']['lora_alpha']}")
print(f"  dropout: {QLORA_CONFIG['lora']['lora_dropout']}")
print(f"\n硬件配置:")
print(f"  设备: {HARDWARE_CONFIG['device']}")
print(f"  GPU内存: {HARDWARE_CONFIG['gpu_memory']} GB")
print("="*60)

配置信息
基座模型: Qwen/Qwen3.6-35B-A3B
模型目录: /home/meerkat/mongoose_ai/models
数据目录: /home/meerkat/mongoose_ai/data
输出目录: /home/meerkat/mongoose_ai/outputs

QLoRA配置:
  rank: 64
  alpha: 128
  dropout: 0.0

硬件配置:
  设备: cuda
  GPU内存: 128 GB


## 1.4 加载基座模型

In [5]:
import os
from utils.training_utils import load_model_and_tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 模型路径 (DGX Spark上可能已预下载到本地)
local_model_path = os.path.join(MODELS_DIR, BASE_MODEL.split("/")[-1])

if os.path.exists(local_model_path):
    model_path = local_model_path
    print(f"使用本地模型: {model_path}")
else:
    model_path = BASE_MODEL
    print(f"从HuggingFace下载: {model_path}")

# 加载模型和分词器
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
)
    
model = AutoModelForCausalLM.from_pretrained(
    "/home/meerkat/mongoose_ai/models/Qwen3.6-35B-A3B",
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
)

print("\n模型加载成功!")
print(model.get_memory_footprint())

使用本地模型: /home/meerkat/mongoose_ai/models/Qwen3.6-35B-A3B


[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/693 [00:00<?, ?it/s]

/home/meerkat/mongoose_ai/venv_v5/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



模型加载成功!
67207562752


## 1.5 测试模型推理

In [7]:
# 简单推理测试
test_prompt = "请简要介绍TRIZ方法论的核心思想:"

inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("模型回复:")
print(response)

# 清理显存
del model
del tokenizer
torch.cuda.empty_cache()
print("\n显存已清理")

NameError: name 'tokenizer' is not defined

---

## 下一步

环境搭建完成！接下来请打开: **02_data_preparation.ipynb** 准备训练数据